In [19]:
%load_ext autoreload
%autoreload 2
from src import SetupBoard, analyze, show, show_interactive, to_frame

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
b = SetupBoard()
b.play("e4", "e5", "Nf3", "Nc6")   # the Ponziani position; edit or delete this line
b

## 2. Run the comparison

| knob | what it does |
|---|---|
| `temp` | pawns. How loosely the engine explores — higher plays more second-best moves. **Changes what the numbers mean.** |
| `horizon` | plies rolled out. A move played after the horizon scores 0, so this sets how far ahead "soon" reaches. **Changes what the numbers mean.** |
| `n` | rollouts per candidate — more is just less noise |
| `depth` | engine search depth per move — cost |
| `multipv` | how many candidate moves the engine returns at each step, i.e. the pool the sampler draws from. `1` makes rollouts deterministic and ignores `temp`. |

`n=30, depth=6` takes about a minute. Drop to `n=10, depth=4` while exploring.

In [21]:
rows, A, B = analyze(b.board, "c3", "Nc3", n=30, horizon=14, depth=6, temp=0.6, seed=0)
print(f"{len(rows)} downstream moves scored after each candidate")

78 downstream moves scored after each candidate


## 3. Boards and scores

Three panels — most unique to A, most common to both, most unique to B. Each board shows its candidate played (gold square) with arrows to the moves listed beside it; each row gives that move's value in **both** lines, on one shared scale.

Two controls:
- **moves shown** — how many moves each list displays
- **score** — what the bars measure:
  - *How soon the move gets played* — `1 / E[own moves until first played]`, with never-played censored at the horizon's last own move + 1. Values bunch up near that floor.
  - *How often the move gets played at all* — the fraction of rollouts containing the move. Spreads out more, but says nothing about timing.

Switching the dropdown is instant — both measures come from the same rollouts. For a static version use `show(b.board, rows, A, B, top=6, metric="speed")`.

In [18]:
# slider = how many moves per list; dropdown = which score to rank by
show_interactive(b.board, rows, A, B, top=6)

## 4. The full table

Sorted by score difference, so moves played soonest after A are at the top and soonest after B at the bottom. The `%` columns are how often each move appeared **at all** within the horizon.

In [5]:
san_a, san_b = b.board.san(A), b.board.san(B)
df = to_frame(rows, san_a, san_b)

df.head(20).style.bar(subset=[f"score diff ({san_a} - {san_b})"], align="zero",
                      color=["#e8734c", "#4c9be8"]) \
                 .format("{:.3f}", subset=[f"score after {san_a}",
                                           f"score after {san_b}",
                                           f"score diff ({san_a} - {san_b})"]) \
                 .format("{:.0f}%", subset=[f"% of rollouts played after {san_a}",
                                            f"% of rollouts played after {san_b}"])

,move,score after c3,score after Nc3,score diff (c3 - Nc3),% of rollouts played after c3,% of rollouts played after Nc3
0,Qa4,0.132,0.000,0.132,30%,0%
1,d3,0.136,0.040,0.096,37%,17%
2,Nd2,0.082,0.008,0.074,37%,3%
3,Ne5,0.075,0.028,0.048,30%,10%
4,Bd3,0.047,0.011,0.036,20%,7%
5,Qc2,0.033,0.000,0.033,13%,0%
6,Qb3,0.031,0.000,0.031,10%,0%
7,Bc6,0.042,0.014,0.028,20%,10%
8,d5,0.085,0.062,0.023,23%,23%
9,c4,0.018,0.000,0.018,7%,0%
